# Fraud Detection — Visualisation

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay

from src.config import DATA_PATH, MODEL_PARAMS, TEST_SIZE, RANDOM_STATE, MIN_PRECISION
from src.data import load_data, split_data
from src.model import train_model, predict_proba
from src.evaluation import find_best_threshold, evaluate_model

In [ ]:
df = load_data()
X_train, X_val, y_train, y_val = split_data(df)
model = train_model(X_train, y_train)
y_proba = predict_proba(model, X_val)
threshold = find_best_threshold(y_val, y_proba)
report, pr_auc = evaluate_model(y_val, y_proba, threshold)

print(f'Threshold : {threshold:.3f}')
print(f'Recall    : {report["1"]["recall"]:.3f}')
print(f'Precision : {report["1"]["precision"]:.3f}')
print(f'PR-AUC    : {pr_auc:.3f}')

## 1. Courbe Precision-Recall

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val, y_proba)
best_idx = (thresholds >= threshold).argmax()

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(recall, precision, label='PR curve')
ax.axhline(MIN_PRECISION, color='red', linestyle='--', label=f'Min precision = {MIN_PRECISION}')
ax.scatter(recall[best_idx], precision[best_idx], color='green', zorder=5, label=f'Seuil = {threshold:.2f}')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Courbe Precision-Recall')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Matrice de confusion

In [ ]:
y_pred = (y_proba >= threshold).astype(int)
cm = confusion_matrix(y_val, y_pred)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Légit.', 'Fraude']).plot(ax=ax, colorbar=False)
ax.set_title(f'Matrice de confusion (seuil = {threshold:.2f})')
plt.tight_layout()
plt.show()

## 3. Distribution des scores

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(y_proba[y_val.values == 0], bins=50, alpha=0.6, label='Légitimes', color='steelblue')
ax.hist(y_proba[y_val.values == 1], bins=50, alpha=0.6, label='Fraudes', color='crimson')
ax.axvline(threshold, color='black', linestyle='--', label=f'Seuil = {threshold:.2f}')
ax.set_xlabel('Score de probabilité')
ax.set_ylabel('Nombre de transactions')
ax.set_title('Distribution des scores')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Importance des variables

In [ ]:
importances = model.feature_importances_
feature_names = X_train.columns.tolist()

sorted_idx = importances.argsort()

fig, ax = plt.subplots(figsize=(8, 10))
ax.barh(range(len(sorted_idx)), importances[sorted_idx], color='steelblue')
ax.set_yticks(range(len(sorted_idx)))
ax.set_yticklabels([feature_names[i] for i in sorted_idx])
ax.set_xlabel("Importance (nombre de splits)")
ax.set_title("Importance des variables — LightGBM")
plt.tight_layout()
plt.show()